# AI + CAD Gripper Hub Optimization
This notebook loads the dataset, visualizes results, ranks designs, and applies ML to guide optimization.

In [ ]:
# Install required library (run once in Colab)
!pip install adjustText

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from adjustText import adjust_text

In [ ]:
# Load dataset (upload file in Colab first)
df = pd.read_csv('gripper_dataset.csv')
print(df.head())

In [ ]:
# Plot: Mass vs Max Stress with overlap handling
plt.figure(figsize=(6,5))
plt.scatter(df['mass'], df['max_stress'])
texts=[]
for i,row in df.iterrows():
    texts.append(plt.text(row['mass'], row['max_stress'], row['version'], fontsize=8))
adjust_text(texts, arrowprops=dict(arrowstyle='->', lw=0.5))
plt.xlabel('Mass (g)')
plt.ylabel('Max Stress (MPa)')
plt.title('Mass vs Stress')
plt.show()

In [ ]:
# Ranking function
def minmax(series):
    return (series - series.min()) / (series.max() - series.min())

df['score'] = (
    0.35*(1-minmax(df['mass'])) +
    0.25*(1-minmax(df['max_stress'])) +
    0.15*(1-minmax(df['max_disp'])) +
    0.25*(minmax(df['safety_factor']))
)

df['rank'] = df['score'].rank(ascending=False)
print(df.sort_values('rank')[['version','score','rank']])

In [ ]:
# ML models
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import mean_absolute_error, r2_score

feature_cols = ['pocket_depth','through_cut','rib_thickness','cutout_length','cutout_angle']
X = df[feature_cols]
y = df['max_stress']

loo = LeaveOneOut()
lin_pred = cross_val_predict(LinearRegression(), X, y, cv=loo)
rf_pred = cross_val_predict(RandomForestRegressor(n_estimators=300, random_state=42), X, y, cv=loo)

print('Linear MAE:', mean_absolute_error(y, lin_pred))
print('RF MAE:', mean_absolute_error(y, rf_pred))